<style>
  /* 1. Fix standard input code cells */
  pre, code, .jp-CodeCell .jp-Cell-inputWrapper {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
  }

  /* 2. Fix display() outputs and rendered Markdown */
  .jp-OutputArea-output, 
  .jp-OutputArea-output pre, 
  .jp-RenderedHTMLCommon pre, 
  .jp-RenderedMarkdown pre {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
      
      /* Forces the browser to print the overflow instead of hiding it in a scrollbox */
      overflow-x: visible !important; 
      max-width: 100% !important;
  }
</style>

In [53]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [60]:
# ============================================================
# KONFIGURASI UTAMA - Ubah nilai di sini sebelum menjalankan
# ============================================================

# Dataset (None = gunakan file pertama di datasets/pci-ml-26/)
FILENAME = None

# Parameter split data
N_SEED = 200
TEST_SIZE = 0.2
SHARE_TOL = 0.01
PCI_MIN = 0

# Parameter tuning Random Forest
TUNING_N_ITER = 100
TUNING_CV_SPLITS = 5

# Model seed (RF, XGBoost, ANN)
SEED = 42

# Memori
RELEASE_MEMORY = True


Parameter: file None | jumlah seed 200 | test size 0.2 | toleransi 0.01 | PCI minimum 0


### **Prediksi *Pavement Condition Index* menggunakan Model *Machine Learning (RF, XGBoost, ANN)***

#### 1. Pembersihan data dan rekayasa fitur

In [69]:
# Memuat dataset
from IPython.display import HTML
from pci_ml.data import list_datasets, load_data, add_aggregate_features, clean_data
from pci_ml.memory import configure
from pci_ml.ui import format_dataframe_table

configure(release=RELEASE_MEMORY)

datasets = list_datasets()
filename = FILENAME if FILENAME is not None else datasets[0]

raw = load_data(filename)

df, features = clean_data(load_data(filename))
df, features = add_aggregate_features(df, features)

print(f"File dimuat: {filename} | jumlah seed: {N_SEED} | "
      f"test size: {TEST_SIZE} | toleransi: {SHARE_TOL} | "
      f"PCI minimum: {PCI_MIN}")
print(f"Data telah dibersihkan; {len(features)} fitur telah dihitung.")
display(HTML(format_dataframe_table(df)))


File dimuat: pci-ml-26.xlsx | jumlah seed: 200 | test size: 0.2 | toleransi: 0.01 | PCI minimum: 0
Data telah dibersihkan; 57 fitur telah dihitung.


ROAD_NAME,ROAD_ID,SEGMENT_ID,GATOR_CRACK_L,GATOR_CRACK_M,…,SEVERITY_WEIGHTED,PCI
Guru Bangkol,R001,S001,0,0,…,0.1672,85.4969
Guru Bangkol,R001,S002,0,0.0039,…,0.1762,73.3368
Guru Bangkol,R001,S003,0,0,…,0.0648,89.9305
Guru Bangkol,R001,S004,0.0014,0.0231,…,0.0644,73.6287
Guru Bangkol,R001,S005,0,0,…,0.0312,96.1874


#### 2. Pemisahan data latih dan data uji

In [ ]:
# Memisah data sesuai kriteria
from pci_ml.data import add_aggregate_features, clean_data, load_data, split_data
from pci_ml.memory import release
from pci_ml.ui import split_progress

if globals().get("df") is None:
    df, features = clean_data(load_data(filename))
    df, features = add_aggregate_features(df, features)

result = split_data(df, features, filename, {
    "n_seed": N_SEED,
    "test_size": TEST_SIZE,
    "share_tol": SHARE_TOL,
    "pci_min": PCI_MIN,
}, progress=split_progress)

print()

# Print the summary outside the progress line so it always shows up
print(result["summary"])

release("result", "df")


TypeError: 'NoneType' object is not callable

#### 3. Pelatihan model *machine learning*

In [58]:
# Mempersiapkan input model
import numpy as np
from IPython.display import Markdown
from pci_ml.data import (AGGREGATE_FEATURES, METADATA, PCI, ROAD_ID,
                         latest_checkpoint, load_checkpoint)
from pci_ml.memory import release

# SEED = 42   # model seed (RF, XGBoost, Keras), separate from the split seed in meta["seed"]

# Memuat checkpoint terbaru untuk file yang dipilih
ckpt_tag = latest_checkpoint(filename)

train_df, test_df, features, meta = load_checkpoint(filename, ckpt_tag)
missing = [name for name in AGGREGATE_FEATURES if name not in features]
if missing:
    raise ValueError(
        "Checkpoint belum memuat fitur agregat. Jalankan ulang Cell 7. "
        f"Kolom hilang: {missing}"
    )

X_train = train_df[features].to_numpy(dtype="float32")
X_test = test_df[features].to_numpy(dtype="float32")
y_train = train_df[PCI].to_numpy(dtype="float32")    # PCI scale 0-100; divide by 100 in the ANN cell
y_test = test_df[PCI].to_numpy(dtype="float32")

groups_train = train_df[ROAD_ID].to_numpy()          # for GroupKFold when tuning
test_ids = test_df[[ROAD_ID, *METADATA]].reset_index(drop=True)

assert all(np.isfinite(a).all() for a in (X_train, X_test, y_train, y_test)), "NaN/inf found"

del train_df, test_df

n_tr, n_te = len(X_train), len(X_test)
total = n_tr + n_te
split_text = (f"**Train:** {X_train.shape} ({n_tr / total:.2%}) | "
              f"**Test:** {X_test.shape} ({n_te / total:.2%})")

# Staleness guard: the newest split may have been made with different parameters
current = {"n_seed": N_SEED, "test_size": TEST_SIZE,
           "share_tol": SHARE_TOL, "pci_min": PCI_MIN}
if current != meta["params"]:
    print("PERINGATAN: split terbaru dibuat dengan parameter berbeda. "
          "Jalankan ulang Cell 7 untuk memisahkan data dengan parameter saat ini.")
    print(f"  Parameter saat ini : {current}")
    print(f"  Parameter split    : {meta['params']}")

display(Markdown(f"**Split terpilih:** `{ckpt_tag}`\n\n{split_text}"))

release("df", "ckpt_tag", "current")


**Split terpilih:** `n200_t0.2_s0.01_p0`

**Train:** (933, 57) (79.27%) | **Test:** (244, 57) (20.73%)

In [59]:
# Tuning hyperparameter dan pelatihan model Random Forest final
from IPython.display import HTML, Markdown, display
from pci_ml.experiments import evaluate_oof, evaluate_predictions
from pci_ml.random_forest import tune_random_forest
from pci_ml.ui import build_hyperparameter_table, format_table

RF_SEARCH_SPACE = {
    "n_estimators": [300],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 2],
    "max_features": [0.1, 0.15, 0.2, 0.25, 0.3, 0.4],
    "max_samples": [0.5, 0.6, 0.7, 0.8, 0.9],
}

print(f"Tuning Random Forest: {TUNING_N_ITER} iterasi | {TUNING_CV_SPLITS} fold CV")
rf_result = tune_random_forest(
    X_train, y_train, groups_train,
    RF_SEARCH_SPACE,
    n_iter=TUNING_N_ITER,
    cv_splits=TUNING_CV_SPLITS,
    random_state=SEED,
)

rf_best_params = rf_result["params"]
tuned_model = rf_result["model"]
rf_cv = evaluate_oof(
    y_train, rf_result["oof"], groups_train, rf_result["folds"]
)["pooled"]
tuned_test = evaluate_predictions(y_test, tuned_model.predict(X_test))

param_explanations = {
    "n_estimators": "Jumlah pohon keputusan.",
    "max_depth": "Kedalaman maksimum tiap pohon.",
    "min_samples_split": "Jumlah sampel minimum agar sebuah node boleh dipecah.",
    "min_samples_leaf": "Jumlah sampel minimum di tiap daun.",
    "max_features": "Fraksi fitur yang dipertimbangkan di tiap pohon.",
    "max_samples": "Fraksi data latih yang dipakai untuk tiap pohon.",
}

param_table = build_hyperparameter_table(
    rf_best_params, RF_SEARCH_SPACE.keys(), param_explanations
)

evaluation_table = format_table(
    ["Evaluasi", "MAE", "RMSE", "R²"],
    [
        ["Cross-Validation", f"{rf_cv['mae']:.4f}", f"{rf_cv['rmse']:.4f}", f"{rf_cv['r2']:.4f}"],
        ["Test", f"{tuned_test['mae']:.4f}", f"{tuned_test['rmse']:.4f}", f"{tuned_test['r2']:.4f}"],
    ],
    aligns=["left", "right", "right", "right"],
)

n_test_roads = test_ids["ROAD_ID"].nunique()

display(Markdown("### Hyperparameter terbaik"))
display(HTML(param_table))
display(Markdown("### Hasil evaluasi"))
display(HTML(evaluation_table))
# display(Markdown(explanation))


Tuning Random Forest: 100 iterasi | 5 fold CV


### Hyperparameter terbaik

Hyperparameter,Nilai,Keterangan
n_estimators,300,Jumlah pohon keputusan.
max_depth,30,Kedalaman maksimum tiap pohon.
min_samples_split,4,Jumlah sampel minimum agar sebuah node boleh dipecah.
min_samples_leaf,1,Jumlah sampel minimum di tiap daun.
max_features,0.4,Fraksi fitur yang dipertimbangkan di tiap pohon.
max_samples,0.9,Fraksi data latih yang dipakai untuk tiap pohon.


### Hasil evaluasi

Evaluasi,MAE,RMSE,R²
Cross-Validation,5.8587,9.0042,0.8455
Test,4.7544,7.9150,0.8724


### *Source code* modul dan fungsi

- `src/pci_ml/data.py` - pembersihan data, rekayasa fitur agregat, split, dan checkpoint.
- `src/pci_ml/random_forest.py` - factory estimator dan tuning Random Forest.
- `src/pci_ml/experiments.py` - fold, OOF, metrik, bobot, dan kalibrasi yang model-agnostic.
- `src/pci_ml/ui.py` - tabel HTML dan progress split.
- `src/pci_ml/memory.py` - manajemen memori notebook.